# Pyrates FEP
# Notebook 04: Full campaign, game assignment

Formalizes the full campaign's game assignment (8 experts) for handoff to the annotation
platform. The expert names produced by notebook 02 (`Michael`, `Magic`, ...) are pseudonyms,
same principle as the pilot's `Penny`/`Shaquille`, and are kept as-is here.

### Inputs:
- `data/output/expert_assignment.csv` (notebook 02's output)

### Outputs:
- `campaign/2_full/full_campaign_assignment.csv`: same assignment, handed to the
  annotation platform alongside the coding form.

## 1. Setup and imports

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root -> data/, campaign/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CAMPAIGN_DIR = PROJECT_ROOT / "campaign" / "2_full"

INPUT_PATH = DATA_DIR / "output" / "expert_assignment.csv"
OUTPUT_PATH = CAMPAIGN_DIR / "full_campaign_assignment.csv"

# Notebook 02's EXPERT_CAPS order (highest workload cap first), kept here only for a
# stable display order - the pseudonyms themselves are not modified.
EXPERTS = ["Michael", "Magic", "Larry", "Charles", "Scottie", "Patrick", "David", "Clyde"]

print("Project root :", PROJECT_ROOT.name)
print("Input        :", INPUT_PATH.relative_to(PROJECT_ROOT), "| exists:", INPUT_PATH.exists())
print("Output       :", OUTPUT_PATH.relative_to(PROJECT_ROOT))

## 2. Load and check the assignment

In [ ]:
assignment = pd.read_csv(INPUT_PATH, dtype=str)
assignment["n_received"] = assignment["n_received"].astype(int)

unknown = (set(assignment["expert_1"]) | set(assignment["expert_2"])) - set(EXPERTS)
assert not unknown, f"EXPERTS is missing: {unknown}"

# Every game assigned to exactly 2 distinct experts, and only once
assert (assignment["expert_1"] != assignment["expert_2"]).all()
assert not assignment["game_id"].duplicated().any()

print(f"Games: {len(assignment)} | Experts: {len(EXPERTS)}")
assignment.sort_values("game_id")

## 3. Per-expert game lists

In [ ]:
for expert in EXPERTS:
    is_expert = (assignment["expert_1"] == expert) | (assignment["expert_2"] == expert)
    rows = assignment.loc[is_expert].sort_values("game_id")
    n_feedback = int(rows["n_received"].sum())
    print(f"{expert} : {len(rows)} games, {n_feedback} feedback instances : {';'.join(rows['game_id'].tolist())}")

## 4. Output

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
assignment.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
print(f"Written: {OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({len(assignment)} rows)")

## 5. Coding on the FEP platform

### Inputs:
- Assignment from part 4: `campaign/2_full/full_campaign_assignment.csv`
- Coding form: `campaign/2_full/full_form.json`

The 8 experts code their assigned games on the FEP platform, same as the pilot. No code
here: the coding itself happens on the platform, outside this notebook.

### Output:
`campaign/2_full/full_result.csv`, one row per (game, trace, expert) with the expert's
answers to the form, same format as `campaign/1_pilot/pilot_result.csv`. Analyzed in
notebook 05.